# 01 数据接入与体检

**目标**：把 Olist 9 张原始 CSV 读进来，确认行数 / 列数 / 日期类型正确，为后续分析打底。

- 数据位置：`data_raw/`
- 约定：日期列统一转 `datetime64`；`products` 的拼写错误列名（`lenght`）已修正
- 全项目统一分析窗口：`2017-01-01 ~ 2018-08-31`（本 notebook 不套窗口，只做接入）

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
# 从当前目录向上找 data_raw/，兼容从 notebooks/ 或项目根启动
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data_raw").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data_raw"
assert RAW_DIR.exists(), f"找不到 data_raw/，当前 PROJECT_ROOT = {PROJECT_ROOT}"
print("PROJECT_ROOT =", PROJECT_ROOT)
print("RAW_DIR      =", RAW_DIR)

PROJECT_ROOT = C:\Users\iiLI\Desktop\1
RAW_DIR      = C:\Users\iiLI\Desktop\1\data_raw


## 1. 文件清单与读取配置

In [3]:
FILES = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

DATE_COLS = {
    "orders": [
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}

# 原 CSV 列名拼写错误，统一修正（和 SQL DDL 对齐）
PRODUCTS_RENAME = {
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
}

## 2. 读取 + 类型清洗

In [5]:
def load_raw() -> dict[str, pd.DataFrame]:
    """读取 9 张原始 CSV，并把日期列转成 datetime。"""
    dfs: dict[str, pd.DataFrame] = {}
    for name, fname in FILES.items():
        df = pd.read_csv(RAW_DIR / fname)
        if name == "products":
            df = df.rename(columns=PRODUCTS_RENAME)
        for col in DATE_COLS.get(name, []):
            df[col] = pd.to_datetime(df[col], errors="coerce")
        dfs[name] = df
        print(f"{name:22s} rows={df.shape[0]:>8,}  cols={df.shape[1]:>2}")
    return dfs

## 3. 执行并体检

In [6]:
dfs = load_raw()

orders                 rows=  99,441  cols= 8
customers              rows=  99,441  cols= 5
order_items            rows= 112,650  cols= 7
order_payments         rows= 103,886  cols= 5
order_reviews          rows=  99,224  cols= 7
products               rows=  32,951  cols= 9
sellers                rows=   3,095  cols= 4
geolocation            rows=1,000,163  cols= 5
category_translation   rows=      71  cols= 2


### 行数 / 列数总览

In [7]:
summary = pd.DataFrame(
    [{"table": k, "rows": v.shape[0], "cols": v.shape[1]} for k, v in dfs.items()]
).sort_values("table").reset_index(drop=True)
summary

,table,rows,cols
0,category_translation,71,2
1,customers,99441,5
2,geolocation,1000163,5
3,order_items,112650,7
4,order_payments,103886,5
5,order_reviews,99224,7
6,orders,99441,8
7,products,32951,9
8,sellers,3095,4


### 关键类型检查（orders 的 5 个日期列应为 datetime64）

In [8]:
dfs["orders"].dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object